In [ ]:
pip install pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.5 MB/s eta 0:00:00


In [ ]:
from PyPDF2 import PdfReader

reader = PdfReader("/content/sample_data/Letter of Admission_53283609.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

1
ZULASSUNGSBES TÄTIGUNG / LETTER OF ADMISSION
Potsdam,10/09/2025
Sehrgeehrte/r Abhishek GowdaThammanna Gowda,
wirfreuenuns,hiermitbestätigen zukönnen,dassSiedienotwendigen Hochschulzugangsvoraussetzungen erfüllenundfürdas
StudiumanderUniversity ofEuropeforAppliedSciencesGmbHakzeptiert wurden.
DieUniversity ofEuropeforAppliedSciencesGmbHisteinedeutsche staatlichanerkannte Privathochschule mitSitzinPotsdam und
weiteren Standorten inIserlohn, Hamburg undBerlin.InBezugaufdasLehrprogramm unddemfürHochschulen geltenden
Rechtsrahmen unterliegt dieHochschule demHochschulgesetz derLandesregierung Brandenburg.
Wearepleasedtoconfirmthatyouhavefulfilledallnecessary admission requirements andhavebeenaccepted toourinstitution, the
University ofEuropeforAppliedSciencesGmbH,intheabove-mentioned programme.
TheUniversity ofEuropeforAppliedSciencesGmbHisaGerman, state-recognized privateuniversity basedinPotsdam withfurther
locations inIserlohn,Hamburg andBerlin.Withregardtotheteachingprogramme andtheleg

In [ ]:
import re

# def clean_text(text):
#     # add space before capital letters (German/English mix)
#     text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)

#     # fix multiple spaces
#     text = re.sub(r'\s+', ' ', text)

#     return text.strip()

# cleaned_text = clean_text(text)

def better_clean(text):
    import re

    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9€., ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

cleaned_text = better_clean(text)

print(cleaned_text[:500])

1 ZULASSUNGSBES T TIGUNG LETTER OF ADMISSION Potsdam,10 09 2025 Sehrgeehrte r Abhishek Gowda Thammanna Gowda, wirfreuenuns,hiermitbest tigen zuk nnen,dass Siedienotwendigen Hochschulzugangsvoraussetzungen erf llenundf rdas Studiumander University of Europefor Applied Sciences Gmb Hakzeptiert wurden. Die University of Europefor Applied Sciences Gmb Histeinedeutsche staatlichanerkannte Privathochschule mit Sitzin Potsdam und weiteren Standorten in Iserlohn, Hamburg und Berlin.In Bezugaufdas Lehrpr


In [ ]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


chunks = chunk_text(cleaned_text,30)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])

Total chunks: 9

First chunk:
 1 ZULASSUNGSBES T TIGUNG LETTER OF ADMISSION Potsdam,10 09 2025 Sehrgeehrte r Abhishek Gowda Thammanna Gowda, wirfreuenuns,hiermitbest tigen zuk nnen,dass Siedienotwendigen Hochschulzugangsvoraussetzungen erf llenundf rdas Studiumander University of Europefor Applied


In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(chunks)

In [ ]:
print(len(embeddings))
print(embeddings[0].shape)

9
(384,)


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.1 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

embedding_array = np.array(embeddings).astype('float32')

dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)

print("Total vectors:", index.ntotal)
print("Dimension:", dimension)
print(embedding_array)

Total vectors: 9
Dimension: 384
[[-0.01832672  0.03913769  0.04868596 ... -0.07124022 -0.02549044
   0.0221127 ]
 [-0.03122171 -0.0099094   0.0491443  ... -0.00247587 -0.03016992
   0.0253663 ]
 [ 0.02187397  0.05012226  0.07299049 ... -0.05572964 -0.03405911
   0.04677951]
 ...
 [-0.04049528  0.04911388  0.05276505 ... -0.10706779 -0.05443116
  -0.05469557]
 [-0.09457774 -0.01479509 -0.00338238 ... -0.14556414 -0.005902
  -0.00927107]
 [-0.06753001  0.02896086  0.02618526 ... -0.03641319  0.08313793
  -0.00148833]]


In [ ]:
#query = "course start date start of studies start of lectures"

query_vec = model.encode([query]).astype('float32')

D, I = index.search(query_vec, k=3)

print("Top matching chunks:\n")

for i in I[0]:
    print(chunks[i])
    print("------")

Top matching chunks:

Academic programme Data Science,M.Sc. Unterrichtssprache Medium ofinstruction English German language skillsnotrequired Studienform Attendance type Full Time Studiendauer Regular numbers ofsemesters 3semester Semesterstart Start ofstudies 01.09.2025 Vorlesungsbeginn Start oflectures 29.09.2025 Letzter
------
1 ZULASSUNGSBES T TIGUNG LETTER OF ADMISSION Potsdam,10 09 2025 Sehrgeehrte r Abhishek Gowda Thammanna Gowda, wirfreuenuns,hiermitbest tigen zuk nnen,dass Siedienotwendigen Hochschulzugangsvoraussetzungen erf llenundf rdas Studiumander University of Europefor Applied
------
Immatrikulationstag Last Dayof Enrolment 10.10.2025 Voraussichtlicher Abschluss des Studiums expected completion ofstudies.28.02.2027 Standort University Campus Potsdam J hrliche Programmgeb hren Annual Tution Fees €12,232.00 Semesterticketgeb hr Semester Ticket Fee asof10
------


In [ ]:
context = "\n".join(chunks)

print(context[:500])

1 ZULASSUNGSBES T TIGUNG LETTER OF ADMISSION Potsdam,10 09 2025 Sehrgeehrte r Abhishek Gowda Thammanna Gowda, wirfreuenuns,hiermitbest tigen zuk nnen,dass Siedienotwendigen Hochschulzugangsvoraussetzungen erf llenundf rdas Studiumander University of Europefor Applied
Sciences Gmb Hakzeptiert wurden. Die University of Europefor Applied Sciences Gmb Histeinedeutsche staatlichanerkannte Privathochschule mit Sitzin Potsdam und weiteren Standorten in Iserlohn, Hamburg und Berlin.In Bezugaufdas Lehrpr


In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 model and tokenizer
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use top 3 chunks from FAISS
top_chunks = [chunks[i] for i in I[0]]
context = "\n".join(top_chunks)

query = "sumarize?"

prompt = f"""Answer the question based on the context.

Context: {context}

Question: {query}

Answer:"""

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode the answer
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


a
